#Librerias y ETL

In [1]:
import pandas as pd
import os
import numpy as np

fold = "/content/datos/"
files = os.listdir(fold)
files = sorted(files) #Periodo 2006 - 2017
dfs = []

for file in files:
    if file.endswith(".csv"):
        print(file)
        try:
            df = pd.read_csv(os.path.join(fold,file), sep=";", encoding="utf-8")
        except UnicodeDecodeError:
            df = pd.read_csv(os.path.join(fold, file), sep=";", encoding="latin1")

        dfs.append(df)

Resumen_Rendimiento 2002.csv
Resumen_Rendimiento 2003.csv
Resumen_Rendimiento 2004.csv
Resumen_Rendimiento 2005.csv


/tmp/ipykernel_13288/1498661686.py:14: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(fold,file), sep=";", encoding="utf-8")


Resumen_Rendimiento 2006.csv
Resumen_Rendimiento 2007.csv
Resumen_Rendimiento 2008.csv
Resumen_Rendimiento 2009.csv
Resumen_Rendimiento 2010.csv
Resumen_Rendimiento 2011.csv
Resumen_Rendimiento 2012.csv
Resumen_Rendimiento 2013.csv
Resumen_Rendimiento 2014.csv
Resumen_Rendimiento 2015.csv
Resumen_Rendimiento 2016.csv
Resumen_Rendimiento 2017.csv
Resumen_Rendimiento 2018.csv
Resumen_Rendimiento 2019.csv
Resumen_Rendimiento 2020.csv
Resumen_Rendimiento 2021.csv
Resumen_Rendimiento 2022.csv
Resumen_Rendimiento 2023.csv
Resumen_Rendimiento 2024.csv


Debido a que se descargaron por orden, es facil assignarle un año

In [2]:
for i, date_year in enumerate(range(2002,2025)):
  dfs[i]["Date"] = date_year

Total de columnas

In [3]:
suma = 0
for i in range(len(dfs)):
  suma += dfs[i].shape[0]
print(suma)

324631


---
**Enumeracion de Regiones**

Debido a que son datos hisstoricos, hay que considerar los cambios que tuvo la enumeracion de regiones a lo largo del tiempo, estos son:


*   En el 2002 habian 13 Regiones, segun la pagina 45 del [censo del 2002](https://www.memoriachilena.gob.cl/archivos2/pdfs/MC0055471.pdf)
*   El 2007 se creo la region de "Los Ríos" como rregion numero 14
*   Tambien se establecio "Arica y Parinacota" como región numero 15
*   En el 2018 se crea la Región del "Ñuble" numero 16






El Objetivo ahora es:
Identificar los establecimientos que hayan cambiado su numero de region y luego los numeros anteriores se le asignara el numero actual, asi podremos identificar a las escuelas con

Primero identificamos si existen estos cambios

In [4]:
for i in range(len(dfs)):
  temp = dfs[i]
  if 0 in temp["COD_REG_RBD"].unique():
    dfs[i] = temp[temp["COD_REG_RBD"] != 0 ]

#periodo 2002 al 2024
for i in range(len(dfs)):
  var_year = range(2002,2025)[i]
  if var_year == 2007 or var_year == 2017 or var_year == 2024:
    print(f"periodo {var_year} con {len(dfs[i]["COD_REG_RBD"].unique())} regiones")
    pass
  else:
    var_year_plus = range(2002,2025)[i+1]
    temp = dfs[i]
    temp_plus1 = dfs[i+1]
    if (temp["COD_REG_RBD"].unique() == temp_plus1["COD_REG_RBD"].unique()).mean() == 1.0:
      print(f"{var_year} - {var_year_plus}: True")


2002 - 2003: True
2003 - 2004: True
2004 - 2005: True
2005 - 2006: True
2006 - 2007: True
periodo 2007 con 13 regiones
2008 - 2009: True
2009 - 2010: True
2010 - 2011: True
2011 - 2012: True
2012 - 2013: True
2013 - 2014: True
2014 - 2015: True
2015 - 2016: True
2016 - 2017: True
periodo 2017 con 15 regiones
2018 - 2019: True
2019 - 2020: True
2020 - 2021: True
2021 - 2022: True
2022 - 2023: True
2023 - 2024: True
periodo 2024 con 16 regiones


Vemos que todos los años cumplen con la cantidad adecuada de regiones.

---
**Llave Primaria**




Ahora comprobamos los establecimientos que tuvieron cambios en el digito de su región.

In [5]:
dfs_temp = dfs.copy()
for i in range(len(dfs)):
  dfs_temp[i] = dfs[i].reindex(columns=["RBD","Date","NOM_RBD",'COD_REG_RBD','COD_PRO_RBD','COD_COM_RBD','COD_DEPROV_RBD'], fill_value=None)
dfs_temp = pd.concat(dfs_temp,axis=0)


In [6]:
dfs_temp = dfs_temp.drop_duplicates()
dfs_temp_antiguo = dfs_temp.copy()

El siguiente codigo cambia la region del establecimiento por el ultima region registrada (2024) solo a los uqe tienen mas de 1 region en la columna.

Además se modifica las columnas de comuna, departamento y provincia, añadiendo el ultimo valor registrado, ya que tambien existe la repgunta **¿Es posible que un establecimiento haya cambiado de comuna o provincia?**

Tambien el nombre del establecimiento se le acctualizara por el ultimo registro en los caso donde hayan mas de 2 nombres.

In [7]:
for id in dfs_temp["RBD"].unique():
  id_region_max = dfs_temp[dfs_temp["RBD"]==id].sort_values("Date")["COD_REG_RBD"].iloc[-1]
  num_reg = dfs_temp[dfs_temp["RBD"]==id]["COD_REG_RBD"].unique().shape[0]

  id_com_max = dfs_temp[dfs_temp["RBD"]==id].sort_values("Date")["COD_COM_RBD"].iloc[-1]
  num_com = dfs_temp[dfs_temp["RBD"]==id]["COD_COM_RBD"].unique().shape[0]

  id_prov_max = dfs_temp[dfs_temp["RBD"]==id].sort_values("Date")["COD_PRO_RBD"].iloc[-1]
  num_prov = dfs_temp[dfs_temp["RBD"]==id]["COD_PRO_RBD"].unique().shape[0]

  id_deprov_max = dfs_temp[dfs_temp["RBD"]==id].sort_values("Date")["COD_DEPROV_RBD"].iloc[-1]
  num_deprov = dfs_temp[dfs_temp["RBD"]==id]["COD_DEPROV_RBD"].unique().shape[0]

  if num_reg >2:
    print("Establecimiento con más de 2 region")
  elif num_reg>1:
    dfs_temp.loc[dfs_temp["RBD"] == id, "COD_REG_RBD"] = id_region_max

    dfs_temp.loc[dfs_temp["RBD"] == id, "COD_DEPROV_RBD"] = id_deprov_max
    dfs_temp.loc[dfs_temp["RBD"] == id, "COD_COM_RBD"] = id_com_max
    dfs_temp.loc[dfs_temp["RBD"] == id, "COD_PRO_RBD"] = id_prov_max

  elif num_reg == 0:
    print("Establecimiento sin region")

  id_NOM_max = dfs_temp[dfs_temp["RBD"]==id].sort_values("Date")["NOM_RBD"].iloc[-1]
  num_NOM = dfs_temp[dfs_temp["RBD"]==id]["NOM_RBD"].unique().shape[0]
  if num_NOM>1:
    dfs_temp.loc[dfs_temp["RBD"] == id, "NOM_RBD"] = id_NOM_max


Comprobemos

Comprobamos con el dataset copiado antes del cambio.

In [8]:
dfs_temp_antiguo[dfs_temp["RBD"]==1]

,RBD,Date,NOM_RBD,COD_REG_RBD,COD_PRO_RBD,COD_COM_RBD,COD_DEPROV_RBD
0,1,2002,LICEO POLITECNICO,1,NaN,1201,NaN
0,1,2003,LICEO POLITECNICO,1,NaN,1201,NaN
0,1,2004,LICEO POLITECNICO ARICA,1,NaN,1201,NaN
0,1,2005,LICEO POLITECNICO ARICA,1,NaN,1201,NaN
0,1,2006,LICEO POLITECNICO ARICA,1,NaN,1201,NaN
0,1,2007,LICEO POLITECNICO ARICA,1,NaN,1201,NaN
0,1,2008,LICEO POLITECNICO ARICA,15,NaN,15101,NaN
0,1,2009,LICEO POLITECNICO ARICA,15,NaN,15101,NaN
0,1,2010,LICEO POLITECNICO ARICA,15,NaN,15101,NaN
0,1,2011,LICEO POLITECNICO ARICA,15,NaN,15101,NaN


In [9]:
dfs_temp[dfs_temp["RBD"]==1]

,RBD,Date,NOM_RBD,COD_REG_RBD,COD_PRO_RBD,COD_COM_RBD,COD_DEPROV_RBD
0,1,2002,LICEO POLITECNICO ARICA,15,151.0,15101,151.0
0,1,2003,LICEO POLITECNICO ARICA,15,151.0,15101,151.0
0,1,2004,LICEO POLITECNICO ARICA,15,151.0,15101,151.0
0,1,2005,LICEO POLITECNICO ARICA,15,151.0,15101,151.0
0,1,2006,LICEO POLITECNICO ARICA,15,151.0,15101,151.0
0,1,2007,LICEO POLITECNICO ARICA,15,151.0,15101,151.0
0,1,2008,LICEO POLITECNICO ARICA,15,151.0,15101,151.0
0,1,2009,LICEO POLITECNICO ARICA,15,151.0,15101,151.0
0,1,2010,LICEO POLITECNICO ARICA,15,151.0,15101,151.0
0,1,2011,LICEO POLITECNICO ARICA,15,151.0,15101,151.0


In [10]:
dfs_temp_antiguo = dfs_temp_antiguo.drop(columns=["Date"])
dfs_temp = dfs_temp.drop(columns=["Date"])

In [11]:
print("df actualizaado:",dfs_temp.drop_duplicates().shape[0]," y df antiguo:",dfs_temp_antiguo.drop_duplicates().shape[0])

df actualizaado: 31231  y df antiguo: 49944


**Seleccion de columnas**



In [12]:
dfs[20].columns[:30]

Index(['AGNO', 'RBD', 'DGV_RBD', 'NOM_RBD', 'COD_REG_RBD', 'NOM_REG_RBD_A',
       'COD_PRO_RBD', 'COD_COM_RBD', 'NOM_COM_RBD', 'COD_DEPROV_RBD',
       'NOM_DEPROV_RBD', 'COD_DEPE', 'COD_DEPE2', 'RURAL_RBD', 'ESTADO_ESTAB',
       'COD_ENSE', 'COD_ENSE2', 'APR_HOM_01', 'APR_HOM_02', 'APR_HOM_03',
       'APR_HOM_04', 'APR_HOM_05', 'APR_HOM_06', 'APR_HOM_07', 'APR_HOM_08',
       'APR_HOM_TO', 'APR_MUJ_01', 'APR_MUJ_02', 'APR_MUJ_03', 'APR_MUJ_04'],
      dtype='object')

In [13]:
columnas_key = ["RBD","NOM_RBD"]
columnas_regiones = ["RBD","COD_REG_RBD","COD_PRO_RBD","COD_COM_RBD","COD_DEPROV_RBD"]
columnas_info = ["RBD","COD_DEPE","RURAL_RBD","ESTADO_ESTAB","COD_ENSE"]
columnas_totales = [texto for texto in dfs[-1].columns if texto.endswith("TO")]
columnas_asistencia = [texto for texto in dfs[-1].columns if texto.startswith("PROM")]

In [14]:
columnas_df_numeric =  ["RBD","Date"] + columnas_totales + columnas_asistencia



---


**dataset numerico**

In [15]:
def cambio_de_tipo(df_numerico):
  columnas_nan_float = df_numerico[columnas_totales].select_dtypes("float64").columns
  df_numerico[columnas_nan_float] = df_numerico[columnas_nan_float].astype("Int64")
  df_numerico[columnas_asistencia] = df_numerico[columnas_asistencia].apply(lambda col: pd.to_numeric(col.astype(str).str.replace(",", "."),errors="coerce"))
  df_numerico["SI_MUJ_TO"] = pd.to_numeric(df_numerico["SI_MUJ_TO"].replace(" ",np.nan),errors="coerce")
  df_numerico["RET_MUJ_TO"] = pd.to_numeric(df_numerico["RET_MUJ_TO"].replace(" ",np.nan),errors="coerce")
  print(df_numerico.shape)
  return df_numerico

In [16]:
dfs_temp_numeric = dfs.copy()
for i in range(len(dfs)):
  dfs_temp_numeric[i] = dfs[i].reindex(columns=columnas_df_numeric, fill_value=None)
df_numeric = pd.concat(dfs_temp_numeric,axis=0)

df_numeric = cambio_de_tipo(df_numeric)

(324629, 21)


In [17]:
df_numeric.info()

<class 'pandas.core.frame.DataFrame'>
Index: 324629 entries, 0 to 15955
Data columns (total 21 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   RBD                324629 non-null  int64  
 1   Date               324629 non-null  int64  
 2   APR_SI_TO          115081 non-null  Int64  
 3   APR_HOM_TO         324629 non-null  int64  
 4   APR_MUJ_TO         324629 non-null  int64  
 5   REP_HOM_TO         324629 non-null  int64  
 6   REP_MUJ_TO         324629 non-null  int64  
 7   SI_HOM_TO          199731 non-null  Int64  
 8   SI_MUJ_TO          185848 non-null  float64
 9   TRA_HOM_TO         324629 non-null  int64  
 10  TRA_MUJ_TO         324629 non-null  int64  
 11  RET_HOM_TO         324629 non-null  int64  
 12  RET_MUJ_TO         308705 non-null  float64
 13  PROM_ASIS_APR_HOM  152076 non-null  float64
 14  PROM_ASIS_APR_MUJ  152199 non-null  float64
 15  PROM_ASIS_APR_SI   48 non-null      float64
 16  PROM_ASI



---


**DataSet Llave Primaria**

In [18]:
df_key =  dfs_temp[["RBD","NOM_RBD"]]
df_key = df_key.drop_duplicates()



---


**DataSet Codigos Regiones**

In [19]:
df_cod_reg = dfs_temp[columnas_regiones]
df_cod_reg = df_cod_reg.drop_duplicates()

In [20]:
df_cod_reg[["COD_PRO_RBD","COD_DEPROV_RBD"]] = df_cod_reg[["COD_PRO_RBD","COD_DEPROV_RBD"]].astype("Int64")

In [21]:
df_cod_reg.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31231 entries, 0 to 15954
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   RBD             31231 non-null  int64
 1   COD_REG_RBD     31231 non-null  int64
 2   COD_PRO_RBD     20412 non-null  Int64
 3   COD_COM_RBD     31231 non-null  int64
 4   COD_DEPROV_RBD  11580 non-null  Int64
dtypes: Int64(2), int64(3)
memory usage: 1.5 MB




---


**DataSet Codigos Establecimiento**

**Caso especial** en el archivo del 2010 la columna "RURAL_RBD", en vez registrar datos binarios como dice el documento, agregaron 1 y 2.

Comparando con dfs los establecimientos de años anteriores se tiene que hacer lo siguiente:

* 1 → 0 (Urbano)
* 2 → 1 (Rural)

In [22]:
dfs[8]["RURAL_RBD"].unique()

array([1, 2])

In [23]:
#Cambio en el archivo del 2010
dfs[8]["RURAL_RBD"] = dfs[8]["RURAL_RBD"].replace(1,0).replace(2,1)

In [24]:
dfs_temp_cod_est = dfs.copy()
for i in range(len(dfs)):
  dfs_temp_cod_est[i] = dfs[i].reindex(columns=  columnas_info + ["Date"], fill_value=None)
df_cod_est = pd.concat(dfs_temp_cod_est,axis=0)


In [25]:
df_cod_est["RURAL_RBD"] = df_cod_est["RURAL_RBD"].astype(int)

In [26]:
df_cod_est.info()

<class 'pandas.core.frame.DataFrame'>
Index: 324629 entries, 0 to 15955
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   RBD           324629 non-null  int64  
 1   COD_DEPE      324629 non-null  int64  
 2   RURAL_RBD     324629 non-null  int64  
 3   ESTADO_ESTAB  143159 non-null  float64
 4   COD_ENSE      324629 non-null  int64  
 5   Date          324629 non-null  int64  
dtypes: float64(1), int64(5)
memory usage: 17.3 MB


Debido a que la informacion de este dataset entrega informacion relevante acerca del tipo de educacion de cada fila de rendimiento ("df_numeric") y además tienen la misma cantidad de filas, es preferible juntar este dataset con "numeric", asi optimizamos los joins.

In [27]:
df_numeric = pd.concat([df_numeric,df_cod_est[["COD_DEPE",	"RURAL_RBD",	"ESTADO_ESTAB",	"COD_ENSE"]]],axis=1,)

In [28]:
df_numeric["ESTADO_ESTAB"] = df_numeric["ESTADO_ESTAB"].astype("Int64")



---


**Exportar DataSet**

In [29]:
import csv


In [30]:
df_key.to_csv("Llave.csv",  sep="|",decimal=".",quotechar='"',quoting=csv.QUOTE_MINIMAL,index = False )

In [31]:
df_numeric.to_csv("Numeric.csv",  sep="|",decimal=".",quotechar='"',quoting=csv.QUOTE_MINIMAL,index = False )

In [32]:
df_cod_reg.to_csv("Cod_reg.csv",  sep="|",decimal=".",quotechar='"',quoting=csv.QUOTE_MINIMAL,index = False )

NOTA: tener cuidado con los analisis, en el df de regiones use el ultimo registro para cambiar los anteriores, tambien lo hice para las comunas  y provincias (es lo que mas puede cambiar a nivel local, por lo que prefiero no investigar tanto), entonces es recomendable hacer analisis por region pero no por comuna y provincia.

In [33]:
df_numeric.info()

<class 'pandas.core.frame.DataFrame'>
Index: 324629 entries, 0 to 15955
Data columns (total 25 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   RBD                324629 non-null  int64  
 1   Date               324629 non-null  int64  
 2   APR_SI_TO          115081 non-null  Int64  
 3   APR_HOM_TO         324629 non-null  int64  
 4   APR_MUJ_TO         324629 non-null  int64  
 5   REP_HOM_TO         324629 non-null  int64  
 6   REP_MUJ_TO         324629 non-null  int64  
 7   SI_HOM_TO          199731 non-null  Int64  
 8   SI_MUJ_TO          185848 non-null  float64
 9   TRA_HOM_TO         324629 non-null  int64  
 10  TRA_MUJ_TO         324629 non-null  int64  
 11  RET_HOM_TO         324629 non-null  int64  
 12  RET_MUJ_TO         308705 non-null  float64
 13  PROM_ASIS_APR_HOM  152076 non-null  float64
 14  PROM_ASIS_APR_MUJ  152199 non-null  float64
 15  PROM_ASIS_APR_SI   48 non-null      float64
 16  PROM_ASI

In [34]:
df_key.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13519 entries, 0 to 15954
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   RBD      13519 non-null  int64 
 1   NOM_RBD  13517 non-null  object
dtypes: int64(1), object(1)
memory usage: 316.9+ KB


In [35]:
df_cod_reg.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31231 entries, 0 to 15954
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   RBD             31231 non-null  int64
 1   COD_REG_RBD     31231 non-null  int64
 2   COD_PRO_RBD     20412 non-null  Int64
 3   COD_COM_RBD     31231 non-null  int64
 4   COD_DEPROV_RBD  11580 non-null  Int64
dtypes: Int64(2), int64(3)
memory usage: 1.5 MB
